# Casa de Repuestos – Sistema Avila

Notebook único para trabajar con el backend (Django): modelos, búsqueda de productos, inventario, ventas, reportes.

**Requisito:** Ejecutar desde la raíz del proyecto (`Avila/`) o desde `backend/`. Tener el entorno virtual activado e instalar: `jupyter`, `django`, dependencias del backend.

## 1. Inicializar Django

Ejecutar esta celda primero (una sola vez por sesión).

In [ ]:
import os
import sys

# Ajustar rutas: si estamos en Avila/, el backend está en backend/
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'backend' else os.getcwd()
BACKEND_DIR = os.path.join(ROOT, 'backend') if os.path.isdir(os.path.join(ROOT, 'backend')) else ROOT
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)
os.chdir(BACKEND_DIR)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'backend.settings')
import django
django.setup()

print(f"Django listo. CWD: {os.getcwd()}")

## 2. Imports – Modelos y utilidades

In [ ]:
from django.db.models import Q, Sum, OuterRef, Subquery, Value, IntegerField
from django.db.models.functions import Coalesce

# Productos
from apps.productos.models import Marca, Categoria, ProductoBase, VarianteProducto
from apps.productos.search_utils import search_term_variants, search_words

# Inventario
from apps.inventario.models import Deposito, Stock, MovimientoStock

# Ventas
from apps.ventas.models import Venta, DetalleVenta

# Clientes
from apps.clientes.models import Cliente

# Configuración
from apps.configuracion.models import ConfiguracionManager

print("Imports OK")

## 3. Búsqueda de productos (lógica unificada)

Misma lógica que la API: código exacto primero, luego búsqueda por palabras con tildes.

In [ ]:
def buscar_variantes(search_term, limit=30):
    """
    Busca variantes por texto: código exacto primero, luego por palabras (nombre, marca, etc.).
    Retorna un QuerySet con select_related y anotación de stock.
    """
    qs = VarianteProducto.objects.select_related(
        'producto_base', 'producto_base__marca', 'producto_base__categoria'
    ).filter(activo=True)
    
    search_clean = (search_term or '').strip()
    if not search_clean:
        return qs.none()
    
    # Atajo: un solo token → búsqueda exacta por código
    if ' ' not in search_clean and len(search_clean) <= 80:
        exact = qs.filter(codigo__iexact=search_clean).first()
        if exact:
            qs = qs.filter(pk=exact.pk)
            _anotar_stock(qs)
            return qs[:limit]
    
    # Búsqueda por palabras (todas deben coincidir)
    words = search_words(search_clean, min_word_len=1)
    if not words:
        words = [search_clean]
    q_total = None
    for word in words:
        variants = search_term_variants(word) if len(word) <= 20 else [word, word.lower(), word.capitalize()]
        q_word = Q()
        for term in variants:
            q_word |= (
                Q(codigo__icontains=term) |
                Q(nombre_variante__icontains=term) |
                Q(producto_base__nombre__icontains=term) |
                Q(producto_base__descripcion__icontains=term) |
                Q(producto_base__marca__nombre__icontains=term)
            )
        q_total = q_word if q_total is None else q_total & q_word
    qs = qs.filter(q_total).distinct()
    _anotar_stock(qs)
    return qs.order_by('-fecha_creacion')[:limit]


def _anotar_stock(queryset):
    """Anota stock_actual_anno en el queryset de variantes (in-place)."""
    subq = Stock.objects.filter(
        variante=OuterRef('pk'),
        deposito__activo=True
    ).values('variante').annotate(total=Sum('cantidad')).values('total')[:1]
    queryset = queryset.annotate(
        stock_actual_anno=Coalesce(Subquery(subq), Value(0, output_field=IntegerField()))
    )
    return queryset

In [ ]:
# Ejemplo: buscar variantes
termo = "piston"  # <-- Cambiar término
resultados = list(buscar_variantes(termo, limit=10))
for v in resultados:
    stock = getattr(v, 'stock_actual_anno', None)
    print(f"{v.codigo} | {v.nombre_completo} | stock={stock} | ${v.precio_mostrador}")

## 4. Inventario – Depósitos y stock

In [ ]:
# Depósito principal
dep = Deposito.objects.filter(es_principal=True, activo=True).first()
print(f"Depósito principal: {dep}")

# Stock de una variante en todos los depósitos
variante_id = 1  # <-- Cambiar ID
stocks = Stock.objects.filter(variante_id=variante_id).select_related('deposito')
for s in stocks:
    print(f"  {s.deposito.nombre}: {s.cantidad}")

## 5. Ventas – Consultas rápidas

In [ ]:
from django.utils import timezone
from datetime import timedelta

# Ventas de hoy
hoy = timezone.now().date()
ventas_hoy = Venta.objects.filter(fecha__date=hoy).select_related('cliente', 'usuario')
total_hoy = ventas_hoy.aggregate(s=Sum('total'))['s'] or 0
print(f"Ventas hoy: {ventas_hoy.count()} | Total: ${total_hoy}")

# Últimas 5 ventas
ultimas = Venta.objects.order_by('-fecha')[:5]
for v in ultimas:
    print(f"  #{v.numero} {v.fecha} {v.cliente_nombre or 'S/C'} ${v.total}")

## 6. Configuración

In [ ]:
# Parámetros de inventario
umbral_critico = ConfiguracionManager.obtener('UMBRAL_STOCK_CRITICO', 2)
umbral_bajo = ConfiguracionManager.obtener('UMBRAL_STOCK_BAJO', 5)
print(f"Stock crítico <= {umbral_critico} | Bajo <= {umbral_bajo}")

## 7. Resumen del sistema (celda libre)

Usar esta celda para consultas ad hoc o pruebas.

In [ ]:
# Ejemplos:
# - Marca.objects.count()
# - VarianteProducto.objects.filter(activo=True).count()
# - buscar_variantes("honda", limit=5)
# - Cliente.objects.filter(activo=True).count()